# AT3：PRAGA 在 Kaggle 环境中的技术复现

**论文：** Huang 等，AAAI 2025，PRAGA。  
**数据集：** Human Lymph Node（RNA + ADT），来自官方 Zenodo Data.zip。  
**复现范围：** 按照官方 10x/HLN 流程运行，仅进行运行环境与代码接口兼容性处理。

本 Notebook 保留 PRAGA 原始模型、输入数据、30 轮训练设置和 mclust EEE 聚类读出。为适配当前 Kaggle 环境，处理了稀疏矩阵转换接口、三模态图函数导入名称冲突、AnnData 类型接口，以及 mclust 版本兼容问题。

当前保存的运行记录已完成 30 轮训练、推理和聚类读出。本次运行的 ARI 为 0.260940，NMI 为 0.370346。结果与论文报告值存在差异，因此本次工作定位为官方流程的技术复现，不表述为数值完全复现。


## 0. Kaggle 运行环境设置

- 在 Settings → Accelerator 中选择 GPU。
- 在 Settings → Internet 中开启网络访问，以便克隆官方仓库和下载 Zenodo 数据。
- 输出文件保存至 `/kaggle/working/PRAGA_AT3/`。
- 官方 Zenodo 数据压缩包约为 247.6 MB。


In [1]:
import os, sys, json, time, hashlib, subprocess, platform, textwrap, zipfile
from pathlib import Path

ROOT = Path("/kaggle/working/PRAGA_AT3")
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / "PRAGA"
DATA_ARCHIVE = ROOT / "Data.zip"
LOG_DIR = ROOT / "logs"
OUT_DIR = ROOT / "results"
LOG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.18.55+-x86_64-with-glibc2.39


In [2]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before running PRAGA training.")
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


Torch: 2.11.0+cu130
CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


## 1. 克隆 PRAGA 官方仓库并记录版本信息

官方仓库：https://github.com/Xubin-s-Lab/PRAGA

运行时会读取并记录仓库当前 commit，便于追溯代码版本。若需要复查本次结果，应保持该 commit 和下载的数据压缩包不变。


In [3]:
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Xubin-s-Lab/PRAGA.git", str(REPO)],
        check=True
    )
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("PRAGA commit:", commit)


Cloning into '/kaggle/working/PRAGA_AT3/PRAGA'...


PRAGA commit: 4adb11c96fc7ddad800fa1787eadcc8b91b42784


## 2. 安装运行依赖

PRAGA 仓库 README 给出的环境基于较早版本的 Python 3.8。Kaggle 镜像可能使用更新版本的 Python，并预装 PyTorch，因此本 Notebook 不主动替换现有 PyTorch。PRAGA 的官方聚类工具通过 `rpy2` 调用 R 的 `mclust`，本次保留官方聚类读出流程。


In [4]:
# 如尚未安装，则安装 R 和基础 mclust。
rscript = subprocess.run(["bash", "-lc", "command -v Rscript"], capture_output=True, text=True)
if rscript.returncode != 0:
    subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq r-base"], check=True)
else:
    print("Rscript found:", rscript.stdout.strip())

# Python 依赖版本与同目录 requirements.txt 保持一致。
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "scanpy==1.12.5",
    "anndata==0.13.4",
    "rpy2==3.5.17",
    "scikit-misc",
    "tqdm",
    "seaborn",
    "matplotlib",
    "scikit-learn"
], check=True)

import scanpy as sc, anndata, rpy2
print("scanpy:", sc.__version__)
print("anndata:", anndata.__version__)
print("rpy2:", rpy2.__version__)


Rscript found: /usr/bin/Rscript
scanpy: 1.12.5
anndata: 0.13.4
rpy2: 3.5.17


/tmp/ipykernel_51/185498046.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)
/tmp/ipykernel_51/185498046.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", anndata.__version__)


## 2.1 固定官方聚类读出所需的 mclust 版本

本单元用于从头运行 Notebook 时，将 R 包 mclust 固定为 6.1，以保持 PRAGA 官方 EEE 聚类读出流程。

当前归档的训练结果和 provenance.json 记录的 mclust 版本为 6.1。此环境配置单元在保存的 Notebook 中没有独立执行输出；如果需要从头复现，请按顺序执行本单元，再运行后续单元。该单元不会修改 PRAGA 模型结构、训练参数或聚类算法。


In [ ]:
import subprocess

r_code = r'''
if (requireNamespace("mclust", quietly = TRUE)) {
  remove.packages("mclust")
}

install.packages(
  "https://cran.r-project.org/src/contrib/Archive/mclust/mclust_6.1.tar.gz",
  repos = NULL,
  type = "source"
)

library(mclust)
cat("mclust version:", as.character(packageVersion("mclust")), "\\n")
stopifnot(as.character(packageVersion("mclust")) == "6.1")
'''

subprocess.run(["Rscript", "-e", r_code], check=True)


## 3. 下载并校验 PRAGA 官方数据集

Zenodo 记录：https://zenodo.org/records/14591305  
数据文件：`Data.zip`  
预期 MD5：`25cdaef517bd2760ddbd406b2fab0afc`

压缩包包含 Human Lymph Node、模拟数据和小鼠脑数据。本 Notebook 根据 `adata_RNA.h5ad`、`adata_ADT.h5ad` 和 `GT_labels.txt` 文件定位 Human Lymph Node 数据，不需要单独上传其他样本。


In [6]:
if not DATA_ARCHIVE.exists():
    url = "https://zenodo.org/records/14591305/files/Data.zip?download=1"
    subprocess.run(["wget", "-O", str(DATA_ARCHIVE), url], check=True)

md5 = hashlib.md5()
with open(DATA_ARCHIVE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        md5.update(chunk)
actual_md5 = md5.hexdigest()
print("Data.zip MD5:", actual_md5)
if actual_md5 != "25cdaef517bd2760ddbd406b2fab0afc":
    raise RuntimeError("Dataset archive MD5 does not match the Zenodo record.")

extract_dir = ROOT / "archive"
extract_dir.mkdir(exist_ok=True)
if not any(extract_dir.rglob("GT_labels.txt")):
    subprocess.run(["unzip", "-q", str(DATA_ARCHIVE), "-d", str(extract_dir)], check=True)

# Zenodo 压缩包包含多个 PRAGA 数据集。根据官方 HLN 流程所需的三个文件，
# 定位 Human Lymph Node 文件夹：RNA、ADT 和真实聚类标签。
hln_candidates = []
for gt_file in extract_dir.rglob("GT_labels.txt"):
    folder = gt_file.parent
    required = [folder / "adata_RNA.h5ad", folder / "adata_ADT.h5ad", gt_file]
    if all(path.is_file() for path in required):
        hln_candidates.append(folder)

if len(hln_candidates) != 1:
    raise RuntimeError(
        "Could not identify exactly one Human Lymph Node folder containing "
        "adata_RNA.h5ad, adata_ADT.h5ad and GT_labels.txt. Candidates: "
        + str([str(x) for x in hln_candidates])
    )

HLN_DIR = hln_candidates[0]
print("Selected Human Lymph Node data directory:", HLN_DIR)
print("Files:", sorted(p.name for p in HLN_DIR.iterdir() if p.is_file()))


--2026-11-03 16:16:23--  https://zenodo.org/records/14591305/files/Data.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.63, 188.185.40.132, 137.138.121.78, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.63|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 247556014 (236M) [application/octet-stream]
Saving to: ‘/kaggle/working/PRAGA_AT3/Data.zip’

     0K .......... .......... .......... .......... ..........  0%  196K 20m33s
    50K .......... .......... .......... .......... ..........  0%  415K 15m7s
   100K .......... .......... .......... .......... ..........  0%  416K 13m18s
   150K .......... .......... .......... .......... ..........  0%  415K 12m24s
   200K .......... .......... .......... .......... ..........  0%  159M 9m55s
   250K .......... .......... .......... .......... ..........  0%  417K 9m53s
   300K .......... .......... .......... .......... ..........  0% 97.2M 8m28s
   350K .......... .......... .......... .........

Data.zip MD5: 25cdaef517bd2760ddbd406b2fab0afc
Selected Human Lymph Node data directory: /kaggle/working/PRAGA_AT3/archive/HLN
Files: ['GT_labels.txt', 'adata_ADT.h5ad', 'adata_RNA.h5ad']


## 4. Kaggle 运行路径与 GPU 说明

PRAGA 官方 `main.py` 在 CUDA 可用时选择 `cuda:1`。本次 Kaggle 运行环境检测到两张 Tesla T4，训练日志也确认模型实际运行在 `cuda:1`，因此本次没有进行设备索引修改。

如果将 Notebook 转移到仅有一张 GPU 的环境，需要先核对官方代码中的设备索引，并视情况调整为 `cuda:0`。此外，官方 `run.sh` 中 HLN 的 `--file_fold` 路径缺少末尾斜杠，因此本 Notebook 在运行命令中补充了斜杠。这些属于运行环境和路径兼容处理，不涉及模型本身的修改。


In [7]:
# 建立官方代码所需的 Data/HLN 目录结构，不修改原始数据。
DATA_ROOT = REPO / "Data"
HLN_TARGET = DATA_ROOT / "HLN"
HLN_TARGET.mkdir(parents=True, exist_ok=True)
for name in ["adata_RNA.h5ad", "adata_ADT.h5ad", "GT_labels.txt"]:
    source = HLN_DIR / name
    if not source.is_file():
        raise FileNotFoundError(source)
    target = HLN_TARGET / name
    if target.is_symlink() or target.exists():
        if target.is_symlink() and target.resolve() == source.resolve():
            continue
        if target.is_dir():
            raise RuntimeError(f"Expected a file but found a directory: {target}")
        target.unlink()
    target.symlink_to(source)
print("HLN files:", [p.name for p in HLN_TARGET.iterdir()])


HLN files: ['adata_RNA.h5ad', 'adata_ADT.h5ad', 'GT_labels.txt']


## 5. 训练前的数据与聚类依赖检查

本单元检查 RNA 与 ADT 的 spot 数量、spot 标识、空间坐标，以及 R/mclust 是否可用；本单元不执行模型训练。


In [8]:
import anndata
rna = anndata.read_h5ad(HLN_TARGET / "adata_RNA.h5ad", backed="r")
adt = anndata.read_h5ad(HLN_TARGET / "adata_ADT.h5ad", backed="r")
print("RNA:", rna.shape, "ADT:", adt.shape)
if rna.n_obs != adt.n_obs:
    raise RuntimeError("RNA and ADT spot counts differ.")
if list(map(str, rna.obs_names)) != list(map(str, adt.obs_names)):
    raise RuntimeError("RNA and ADT spot order differs.")
print("RNA spatial:", "spatial" in rna.obsm)
print("ADT spatial:", "spatial" in adt.obsm)
rna.file.close(); adt.file.close()

subprocess.run(["Rscript", "-e", "library(mclust); cat(as.character(packageVersion('mclust')))"], check=True)


RNA: (3484, 18085) ADT: (3484, 31)
RNA spatial: True
ADT spatial: True


/tmp/ipykernel_51/4130299130.py:2: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  rna = anndata.read_h5ad(HLN_TARGET / "adata_RNA.h5ad", backed="r")
/tmp/ipykernel_51/4130299130.py:3: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adt = anndata.read_h5ad(HLN_TARGET / "adata_ADT.h5ad", backed="r")


6.1

Package 'mclust' version 6.1
Type 'citation("mclust")' for citing this R package in publications.


CompletedProcess(args=['Rscript', '-e', "library(mclust); cat(as.character(packageVersion('mclust')))"], returncode=0)

## 5.1 PRAGA 预处理代码的 SciPy 稀疏矩阵接口兼容处理

PRAGA 官方 `PRAGA/PRAGA/preprocess.py` 使用 `adata.X.A` 将稀疏矩阵转换为稠密数组。在当前 Kaggle 的 SciPy 环境中，CSR 矩阵不再提供 `.A` 属性，因此 ADT CLR 标准化阶段会因此报错。

兼容写法为 `adata.X.toarray()`。本单元只替换稀疏矩阵转换表达式，不改变数据、模型、损失函数、优化器、训练参数或聚类读出；重复运行时会先检查是否已经完成替换。


In [9]:
from pathlib import Path

preprocess_py = REPO / "PRAGA" / "preprocess.py"
if not preprocess_py.is_file():
    raise FileNotFoundError(f"PRAGA preprocessing file not found: {preprocess_py}")

preprocess_source = preprocess_py.read_text(encoding="utf-8")
old_expr = "adata.X.A"
new_expr = "adata.X.toarray()"

if old_expr in preprocess_source:
    count = preprocess_source.count(old_expr)
    preprocess_source = preprocess_source.replace(old_expr, new_expr)
    preprocess_py.write_text(preprocess_source, encoding="utf-8")
    print(f"Applied SciPy compatibility patch to {count} occurrence(s): .A -> .toarray()")
elif new_expr in preprocess_source:
    print("SciPy compatibility patch already present; no change needed.")
else:
    raise RuntimeError(
        "Expected sparse conversion expression was not found. "
        "Do not modify other code automatically; inspect preprocess.py."
    )

# 确认预处理代码已使用 .toarray()。
matching_lines = [
    line.strip()
    for line in preprocess_py.read_text(encoding="utf-8").splitlines()
    if "seurat_clr" in line or "adata.X.toarray()" in line
]
print("\n".join(matching_lines))


Applied SciPy compatibility patch to 1 occurrence(s): .A -> .toarray()
feat_pca = pca.fit_transform(adata.X.toarray())
def seurat_clr(x):
seurat_clr, 1, (adata.X.toarray() if scipy.sparse.issparse(adata.X) else np.array(adata.X))


## 5.2 解决 PRAGA 中 `construct_neighbor_graph` 的名称作用域冲突

官方 `main.py` 在模块层面导入了双模态 `construct_neighbor_graph`，随后又在 `main()` 内部为三模态 Simulation 分支导入同名函数。由于 Python 的局部变量作用域规则，函数内部的同名导入可能导致前面的 10x 分支无法调用原本的双模态函数，并触发 `UnboundLocalError`。

本单元为 Simulation 分支的三模态函数增加别名，保留 Human Lymph Node 10x 分支使用的双模态函数，同时保留 Simulation 分支的三模态函数。该处理不修改模型、损失函数、数据或训练设置。


In [10]:
from pathlib import Path

main_py = REPO / "main.py"
main_source = main_py.read_text(encoding="utf-8")

old_import = "from PRAGA.preprocess_3M import construct_neighbor_graph"
new_import = "from PRAGA.preprocess_3M import construct_neighbor_graph as construct_neighbor_graph_3m"

old_call = "data = construct_neighbor_graph(adata_omics1, adata_omics2, adata_omics3)"
new_call = "data = construct_neighbor_graph_3m(adata_omics1, adata_omics2, adata_omics3)"

if old_import in main_source:
    main_source = main_source.replace(old_import, new_import, 1)
elif new_import in main_source:
    print("Simulation import alias already applied.")
else:
    raise RuntimeError("Could not find the expected Simulation-only import in main.py.")

if old_call in main_source:
    main_source = main_source.replace(old_call, new_call, 1)
elif new_call in main_source:
    print("Simulation function call alias already applied.")
else:
    raise RuntimeError("Could not find the expected three-modality graph call in main.py.")

main_py.write_text(main_source, encoding="utf-8")

# 确认 10x 分支仍调用原双模态函数，
# 只有 Simulation 分支调用增加别名的三模态函数。
lines = main_source.splitlines()
for i, line in enumerate(lines, start=1):
    if "construct_neighbor_graph" in line:
        print(f"{i}: {line.strip()}")


15: from PRAGA.preprocess import construct_neighbor_graph, lsi
60: data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=args.data_type, Arg=args)
78: data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=args.data_type, Arg=args)
92: data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=args.data_type, Arg=args)
107: data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=args.data_type, Arg=args)
123: from PRAGA.preprocess_3M import construct_neighbor_graph as construct_neighbor_graph_3m
124: data = construct_neighbor_graph_3m(adata_omics1, adata_omics2, adata_omics3)


## 6. 运行 PRAGA 官方 HLN 命令

官方 `run.sh` 中 HLN 实验的参数设置为：

- `data_type=10x`
- `n_clusters=10`
- `init_k=10`
- `KNN_k=20`
- `RNA_weight=5`
- `ADT_weight=5`

官方 `Train_model.py` 对 `data_type='10x'` 设置 30 个训练轮次；主脚本中的训练随机种子为 2024，mclust 工具函数使用的随机种子为 2020。本次复现保持这些设置不变。


In [11]:
import time
start = time.time()
cmd = [
    sys.executable, "main.py",
    "--file_fold", str(HLN_TARGET) + "/",  # main.py 使用字符串拼接路径，因此需要末尾斜杠
    "--data_type", "10x",
    "--n_clusters", "10",
    "--init_k", "10",
    "--KNN_k", "20",
    "--RNA_weight", "5",
    "--ADT_weight", "5",
    "--vis_out_path", str(OUT_DIR / "PRAGA_HLN.png"),
    "--txt_out_path", str(OUT_DIR / "PRAGA_HLN.txt"),
]
print("COMMAND:", " ".join(cmd))
try:
    with open(LOG_DIR / "PRAGA_HLN_train.log", "w") as log:
        subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, check=True)
except subprocess.CalledProcessError:
    print("Training failed. Last part of the log:")
    print((LOG_DIR / "PRAGA_HLN_train.log").read_text(errors="replace")[-12000:])
    raise

elapsed = time.time() - start
print(f"Training + inference wall time: {elapsed:.1f} seconds")
print("Log:", LOG_DIR / "PRAGA_HLN_train.log")


COMMAND: /usr/bin/python3 main.py --file_fold /kaggle/working/PRAGA_AT3/PRAGA/Data/HLN/ --data_type 10x --n_clusters 10 --init_k 10 --KNN_k 20 --RNA_weight 5 --ADT_weight 5 --vis_out_path /kaggle/working/PRAGA_AT3/results/PRAGA_HLN.png --txt_out_path /kaggle/working/PRAGA_AT3/results/PRAGA_HLN.txt
Training + inference wall time: 127.5 seconds
Log: /kaggle/working/PRAGA_AT3/logs/PRAGA_HLN_train.log


## 6.1 AnnData 指标脚本接口兼容处理

当前 AnnData 版本已移除官方 `metric.py` 所引用的内部 `SparseDataset` 类型。本单元将其替换为 AnnData 提供的公开 `CSRDataset` 和 `CSCDataset` 类型，以便官方指标脚本在当前环境中运行。该处理仅涉及类型接口引用，不改变评价指标定义。


In [13]:

import anndata as ad

metric_py = REPO / "metric.py"
src = metric_py.read_text(encoding="utf-8")

old = "ad._core.sparse_dataset.SparseDataset"
new = "ad.abc.CSRDataset, ad.abc.CSCDataset"

if old in src:
    src = src.replace(old, new, 1)
    metric_py.write_text(src, encoding="utf-8")
    print("AnnData compatibility patch applied!")
elif new in src:
    print("Patch already applied!")
else:
    raise RuntimeError("没有找到旧类型定义，请检查 metric.py")

print("AnnData:", ad.__version__)


AnnData compatibility patch applied!
AnnData: 0.13.4


/tmp/ipykernel_51/279147464.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("AnnData:", ad.__version__)


## 7. 计算 ARI、NMI 并检查输出文件

如果官方评价脚本及其所需的真实标签文件存在，则运行官方指标脚本。同时，根据真实标签和预测标签独立计算 ARI、NMI，作为结果核对。


In [14]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import numpy as np
pred_path = OUT_DIR / "PRAGA_HLN.txt"
gt_path = HLN_TARGET / "GT_labels.txt"
if not pred_path.exists():
    raise FileNotFoundError(pred_path)
pred = np.loadtxt(pred_path, dtype=str)
gt = np.loadtxt(gt_path, dtype=str)
if len(pred) != len(gt):
    raise RuntimeError(f"Label length mismatch: pred={len(pred)}, gt={len(gt)}")
ari = float(adjusted_rand_score(gt, pred))
nmi = float(normalized_mutual_info_score(gt, pred, average_method="arithmetic"))
print({"n_spots":len(gt), "ARI":ari, "NMI_arithmetic":nmi})

# 如果官方指标脚本存在，则额外运行并保存其输出。
metric_script = REPO / "cal_matrics.py"
if metric_script.exists():
    official_out = OUT_DIR / "official_metrics.txt"
    subprocess.run([
        sys.executable, str(metric_script),
        "--GT_path", str(gt_path),
        "--our_path", str(pred_path),
        "--save_path", str(official_out)
    ], cwd=REPO, check=True)
    print(official_out.read_text(errors="replace"))


{'n_spots': 3484, 'ARI': 0.260940013960164, 'NMI_arithmetic': 0.3703457230846045}
0 9
1 10
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9}
3484
3484
our         jaccard: 0.250926
our         F_measure: 0.401185
our         Mutual Information: 0.692756
Our         (NMI): 0.370346
Our         (AMI): 0.366223
Our         V-measure: 0.370346
Our         Homogeneity: 0.410643 Completeness: 0.337251
Our         (ARI): 0.260940
Our         (FMI): 0.406955
Our     jaccard: 0.250926
Our     F_measure: 0.401185
Our     Mutual Information: 0.692756
Our     NMI: 0.370346
Our     AMI: 0.366223
Our     V-measure: 0.370346
Our     Homogeneity: 0.410643
Our     Completeness: 0.337251
Our     (ARI): 0.260940
Our     (FMI): 0.406955



## 8. 保存实验来源信息并打包结果

本单元根据实际运行产生的变量写入 `provenance.json`，记录代码版本、数据校验值、运行环境、实验参数和评价指标。结果数值来自本次运行输出，不使用预设或估计值。


In [15]:
import importlib.metadata as im

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
training_device = (
    f"cuda:1 ({torch.cuda.get_device_name(1)}; confirmed by training log)"
    if gpu_count > 1
    else f"cuda:0 ({gpu_name})"
)

provenance = {
    "paper": "PRAGA, AAAI 2025, DOI 10.1609/aaai.v39i1.32010",
    "repository": "https://github.com/Xubin-s-Lab/PRAGA",
    "repository_commit": commit,
    "dataset": "PRAGA Zenodo Data.zip, record 14591305",
    "dataset_md5": actual_md5,
    "dataset_sample": "Human Lymph Node (RNA+ADT)",
    "device": gpu_name,
    "gpu_count": gpu_count,
    "training_device": training_device,
    "device_note": "本次 Kaggle 环境检测到两张 Tesla T4；训练日志显示模型运行在 cuda:1，本次未执行 cuda:1 到 cuda:0 的设备索引修改。",
    "anndata_version": im.version("anndata"),
    "torch": torch.__version__,
    "mclust_version": subprocess.check_output(
        ["Rscript", "-e", "cat(as.character(packageVersion('mclust')))"],
        text=True
    ).strip(),
    "python": sys.version,
    "parameters": {
        "data_type": "10x",
        "n_clusters": 10,
        "init_k": 10,
        "KNN_k": 20,
        "RNA_weight": 5,
        "ADT_weight": 5,
        "epochs": 30,
        "training_seed": 2024,
        "mclust_seed": 2020
    },
    "compatibility_patches": [
        "passed trailing slash on --file_fold path",
        "PRAGA/PRAGA/preprocess.py sparse conversion adata.X.A -> adata.X.toarray() for SciPy compatibility",
        "main.py aliases the Simulation-only construct_neighbor_graph import to avoid shadowing the 10x graph function",
        "R mclust pinned from 6.1.3 to archived CRAN 6.1 after the 6.1.3 EEE readout dimnames error",
        "metric.py replaces the removed AnnData internal type ad._core.sparse_dataset.SparseDataset with public ad.abc.CSRDataset and ad.abc.CSCDataset for the official metric helper"
    ],
    "metrics": {
        "ARI": ari,
        "NMI_arithmetic": nmi
    },
    "elapsed_seconds": elapsed
}

(ROOT / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8"
)
print(json.dumps(provenance, indent=2))


{
  "paper": "PRAGA, AAAI 2025, DOI 10.1609/aaai.v39i1.32010",
  "repository": "https://github.com/Xubin-s-Lab/PRAGA",
  "repository_commit": "4adb11c96fc7ddad800fa1787eadcc8b91b42784",
  "dataset": "PRAGA Zenodo Data.zip, record 14591305",
  "dataset_md5": "25cdaef517bd2760ddbd406b2fab0afc",
  "dataset_sample": "Human Lymph Node (RNA+ADT)",
  "device": "Tesla T4",
  "torch": "2.11.0+cu130",
  "mclust_version": "6.1",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "parameters": {
    "data_type": "10x",
    "n_clusters": 10,
    "init_k": 10,
    "KNN_k": 20,
    "RNA_weight": 5,
    "ADT_weight": 5,
    "epochs": 30,
    "training_seed": 2024,
    "mclust_seed": 2020
  },
  "compatibility_patches": [
    "passed trailing slash on --file_fold path",
    "PRAGA/PRAGA/preprocess.py sparse conversion adata.X.A -> adata.X.toarray() for SciPy compatibility",
    "main.py aliases the Simulation-only construct_neighbor_graph import to avoid shadowing the 10x graph functio

In [16]:
# 将本次运行结果打包，供 AT3 提交使用。
import shutil
evidence_zip = ROOT / "PRAGA_AT3_HLN_evidence.zip"
with zipfile.ZipFile(evidence_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in [OUT_DIR / "PRAGA_HLN.png", OUT_DIR / "PRAGA_HLN.txt", LOG_DIR / "PRAGA_HLN_train.log", ROOT / "provenance.json"]:
        if p.exists():
            z.write(p, arcname=p.relative_to(ROOT))
    if (OUT_DIR / "official_metrics.txt").exists():
        z.write(OUT_DIR / "official_metrics.txt", arcname="results/official_metrics.txt")
print("Evidence archive:", evidence_zip)


Evidence archive: /kaggle/working/PRAGA_AT3/PRAGA_AT3_HLN_evidence.zip


## 9. 提交材料核对

- [ ] Kaggle GPU 环境和依赖配置记录。
- [ ] PRAGA 训练完成记录及空间/UMAP 聚类图。
- [ ] `PRAGA_HLN.txt` 预测聚类标签。
- [ ] 官方评价脚本输出及独立计算的 ARI、NMI。
- [ ] `provenance.json`：代码 commit、数据 MD5、运行环境和参数。
- [ ] 复现说明：记录本次实际进行的兼容性处理。
- [ ] 与论文报告值存在差异时，应如实说明，不将其表述为数值完全复现。
